In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed.
# It is defined by the kaggle/python Docker image.
# For example, here are several helpful packages to load.

import numpy as np   # linear algebra
import pandas as pd  # data processing and Excel file I/O

# Input data files are available in the read-only /kaggle/input/ directory.
# Running this cell will list all files mounted under the input directory.

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20 GB to /kaggle/working/ and it is preserved when
# you save a version using "Save & Run All".


In [ ]:
import numpy as np
import pandas as pd
import random
import os
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, EfficientNetV2S
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, UpSampling2D, Concatenate, Add,
    GlobalAveragePooling2D, Dense, Dropout, BatchNormalization,
    Multiply, Activation, Flatten, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


In [ ]:
# Paths to the augmented dataset uploaded on Kaggle.
# We resolve the BASE path dynamically so that it works regardless of your dataset naming.

import os
possibilities = [
    "/kaggle/input/multimodal/Augmented_Multimodal",
    "/kaggle/input/glaucoma-multimodal/Augmented_Multimodal",
    "/kaggle/input/datasets/kartikeybishnoi/glaucoma-multimodal/Augmented_Multimodal"
]

BASE = None
for p in possibilities:
    if os.path.exists(p):
        BASE = p
        break

if BASE is None:
    # Try finding any directory containing 'Augmented_Multimodal' under /kaggle/input
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'Augmented_Multimodal' in dirs:
            BASE = os.path.join(root, 'Augmented_Multimodal')
            break

if BASE is None:
    # Fallback to default
    BASE = "/kaggle/input/multimodal/Augmented_Multimodal"

print("Using dataset BASE path:", BASE)

FUNDUS_PATH = os.path.join(BASE, "Fundus")    # subfolders: Mild, Moderate, Severe
HVF_PATH    = os.path.join(BASE, "HVF")       # subfolders: Mild, Moderate, Severe
OCT_PATH    = os.path.join(BASE, "OCT")       # subfolders: Mild, Moderate, Severe
RNFL_PATH   = os.path.join(BASE, "RNFL_GCC", "Glaucoma AI.xlsx")



In [ ]:
# Load the RNFL and GCC clinical measurements from the Excel sheet.
# This is the tabular branch of our multimodal model - the actual numbers
# that a clinician reads from the OCT machine printout.

df_rnfl = pd.read_excel(RNFL_PATH)
print("RNFL/GCC Excel loaded. Shape:", df_rnfl.shape)
print("Columns:", df_rnfl.columns.tolist())
df_rnfl.head()


In [ ]:
# Clean and align the severity labels to match class folders.
df_rnfl['Glaucoma_Severity'] = df_rnfl['Glaucoma_Severity'].str.upper()
print("Unique severity labels found:", df_rnfl['Glaucoma_Severity'].unique())


In [ ]:
import os
import re
import pandas as pd

CLASSES = {"Mild": 1, "Moderate": 2, "Severe": 3}
CLINICAL_COLS = [
    'Average_RNFL', 'RNFL_Superior', 'RNFL_Inferior', 'RNFL_Nasal', 'RNFL_Temporal',
    'Average_GCC', 'GCC_Superior', 'GCC_Inferior', 'GCC_Supero nasal', 'GCC_Supero temporal',
    'GCC_Infero nasal', 'GCC_Infero temporal'
]

# Separate original Excel rows by class to perform modulo alignment
excel_by_class = {}
for class_name in CLASSES.keys():
    excel_by_class[class_name] = df_rnfl[df_rnfl['Glaucoma_Severity'] == class_name.upper()].copy()

records = []

for class_name, label in CLASSES.items():
    fundus_folder = os.path.join(FUNDUS_PATH, class_name)
    hvf_folder    = os.path.join(HVF_PATH, class_name)
    oct_folder    = os.path.join(OCT_PATH, class_name)

    if not os.path.exists(fundus_folder): continue
    
    fundus_files = sorted([f for f in os.listdir(fundus_folder) if f.lower().endswith(('.jpg', '.png'))])
    hvf_files    = sorted([f for f in os.listdir(hvf_folder)    if f.lower().endswith(('.jpg', '.png'))])
    oct_files    = sorted([f for f in os.listdir(oct_folder)    if f.lower().endswith(('.jpg', '.png'))])

    n = min(len(fundus_files), len(hvf_files), len(oct_files))
    class_excel = excel_by_class[class_name]
    num_rows = len(class_excel)

    for i in range(n):
        fname = fundus_files[i]
        
        # --- ROBUST STEM EXTRACTION (Prevents Data Leakage & Crashes) ---
        stem = fname
        if fname.startswith('aug_'):
            # Safely removes 'aug_' and any numbers like '0066_' from the front
            stem = re.sub(r'^aug_(\d+_)?', '', fname)
        # ----------------------------------------------------------------
        
        row_idx = i % num_rows
        excel_row = class_excel.iloc[row_idx]

        record = {
            "fundus_path": os.path.join(fundus_folder, fundus_files[i]),
            "hvf_path":    os.path.join(hvf_folder,    hvf_files[min(i, len(hvf_files)-1)]),
            "oct_path":    os.path.join(oct_folder,    oct_files[min(i, len(oct_files)-1)]),
            "label":       label,
            "class_name":  class_name,
            "orig_stem":   stem,
        }
        for col in CLINICAL_COLS:
            record[col] = excel_row[col]

        records.append(record)

df = pd.DataFrame(records)
print("Combined dataframe shape:", df.shape)
print(df['class_name'].value_counts())
df.head()


In [ ]:
# Show Fundus, HVF, and OCT images side by side for each severity class.

classes = list(CLASSES.keys())
fig, axes = plt.subplots(len(classes), 3, figsize=(15, 4 * len(classes)))

for row, cls in enumerate(classes):
    sample = df[df["class_name"] == cls].sample(1, random_state=42).iloc[0]

    f_img = cv2.cvtColor(cv2.imread(sample["fundus_path"]), cv2.COLOR_BGR2RGB)
    h_img = cv2.cvtColor(cv2.imread(sample["hvf_path"]),    cv2.COLOR_BGR2RGB)
    o_img = cv2.cvtColor(cv2.imread(sample["oct_path"]),    cv2.COLOR_BGR2RGB)

    axes[row, 0].imshow(f_img); axes[row, 0].set_title(f"Fundus - {cls}"); axes[row, 0].axis("off")
    axes[row, 1].imshow(h_img); axes[row, 1].set_title(f"HVF - {cls}");    axes[row, 1].axis("off")
    axes[row, 2].imshow(o_img); axes[row, 2].set_title(f"OCT - {cls}");    axes[row, 2].axis("off")

plt.suptitle("Sample images from each severity class (all modalities)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x="class_name",
                   order=["Mild", "Moderate", "Severe"],
                   palette="muted")
ax.set_title("Class distribution after augmentation")
ax.set_xlabel("Severity")
ax.set_ylabel("Number of image pairs")
for p in ax.patches:
    ax.annotate(int(p.get_height()),
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 16    # keep small - loading multiple images per sample
EPOCHS     = 100   # will stop early via EarlyStopping

# ── Split logic — uses the file naming from generate_multimodal_dataset.py ──
# File naming convention (set during augmentation):
#   test_XXXX.jpg              — test patient, NEVER augmented
#   orig_trXXXX.jpg            — train patient original (XXXX = position in train array, 0-based)
#   aug_XXXX_trainYYYY.jpg     — augmented from train position YYYY
#
# LEAKAGE GUARANTEE:
#   Test patients split out BEFORE augmentation in generate_multimodal_dataset.py (seed=42).
#   The 'test_' prefix is completely separate from 'orig_tr' and 'aug_' prefixes.
#   A leakage audit confirms 0 aug files derive from test patients.

records_train, records_val, records_test = [], [], []

for class_name, label in CLASSES.items():
    fundus_folder = os.path.join(FUNDUS_PATH, class_name)
    hvf_folder    = os.path.join(HVF_PATH,    class_name)
    oct_folder    = os.path.join(OCT_PATH,    class_name)

    if not os.path.exists(fundus_folder):
        continue

    all_fundus = sorted([
        f for f in os.listdir(fundus_folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    # Partition by prefix — strictly separate
    test_files  = [f for f in all_fundus if f.startswith('test_')]    # test patients, never augmented
    orig_files  = [f for f in all_fundus if f.startswith('orig_tr')]  # train originals (orig_trXXXX)
    aug_files   = [f for f in all_fundus if f.startswith('aug_')]     # train augmented (aug_NNNN_trainXXXX)

    # Validation = 12.5% of orig_tr files (stratified, deterministic)
    random.seed(42)
    random.shuffle(orig_files)
    n_val            = max(1, int(round(len(orig_files) * 0.125)))
    val_files_list   = orig_files[:n_val]
    train_orig_files = orig_files[n_val:]

    def make_record(fname, split_tag):
        # Get matching HVF and OCT by same filename
        hvf_path_f = os.path.join(hvf_folder, fname)
        oct_path_f = os.path.join(oct_folder, fname)
        # Fallback: if exact filename doesn’t exist in HVF/OCT, use the first available
        if not os.path.exists(hvf_path_f):
            hvfs = sorted([f for f in os.listdir(hvf_folder) if f.startswith(fname[:8])])
            hvf_path_f = os.path.join(hvf_folder, hvfs[0]) if hvfs else os.path.join(hvf_folder, sorted(os.listdir(hvf_folder))[0])
        if not os.path.exists(oct_path_f):
            octs = sorted([f for f in os.listdir(oct_folder) if f.startswith(fname[:8])])
            oct_path_f = os.path.join(oct_folder, octs[0]) if octs else os.path.join(oct_folder, sorted(os.listdir(oct_folder))[0])
        return {
            'fundus_path': os.path.join(fundus_folder, fname),
            'hvf_path':    hvf_path_f,
            'oct_path':    oct_path_f,
            'label':       label,
            'class_name':  class_name,
            'split':       split_tag,
            'orig_stem':   fname,
        }

    for f in test_files:
        r = make_record(f, 'test')
        # test_XXXX.jpg — XXXX is raw patient index in source list
        raw_idx = int(f[5:9])
        for col in CLINICAL_COLS:
            r[col] = excel_by_class[class_name].iloc[raw_idx % len(excel_by_class[class_name])][col]
        records_test.append(r)

    for f in val_files_list:
        r = make_record(f, 'val')
        # orig_trXXXX.jpg — XXXX is position in train array
        pos_idx = int(f[7:11])
        for col in CLINICAL_COLS:
            r[col] = excel_by_class[class_name].iloc[pos_idx % len(excel_by_class[class_name])][col]
        records_val.append(r)

    for f in train_orig_files + aug_files:
        r = make_record(f, 'train')
        if f.startswith('aug_'):
            # aug_NNNN_trainYYYY.jpg — YYYY is position in train array
            try:
                pos_idx = int(f.split('_train')[1][:4])
            except Exception:
                pos_idx = 0
        else:
            # orig_trXXXX.jpg — XXXX is position in train array
            pos_idx = int(f[7:11])
        for col in CLINICAL_COLS:
            r[col] = excel_by_class[class_name].iloc[pos_idx % len(excel_by_class[class_name])][col]
        records_train.append(r)

train_df = pd.DataFrame(records_train).reset_index(drop=True)
val_df   = pd.DataFrame(records_val).reset_index(drop=True)
test_df  = pd.DataFrame(records_test).reset_index(drop=True)

print(f"Training set:   {len(train_df)} samples  | class dist: {train_df['class_name'].value_counts().to_dict()}")
print(f"Validation set: {len(val_df)} samples  | class dist: {val_df['class_name'].value_counts().to_dict()}")
print(f"Test set:       {len(test_df)} samples  | class dist: {test_df['class_name'].value_counts().to_dict()}")
print("\nLeakage check:")
print(f"  Test files starting with 'test_':   {(test_df['fundus_path'].str.contains('/test_')).all()}")
print(f"  Val files starting with 'orig_':    {(val_df['fundus_path'].str.contains('/orig_')).all()}")
print(f"  Any aug_ files in test set:         {(test_df['fundus_path'].str.contains('/aug_')).any()}")
print("All checks must be True / False / False respectively.")

X_train_img = train_df[['fundus_path', 'hvf_path', 'oct_path']].values
X_val_img   = val_df[['fundus_path', 'hvf_path', 'oct_path']].values
X_test_img  = test_df[['fundus_path', 'hvf_path', 'oct_path']].values

X_train_tab = train_df[CLINICAL_COLS].values.astype(float)
X_val_tab   = val_df[CLINICAL_COLS].values.astype(float)
X_test_tab  = test_df[CLINICAL_COLS].values.astype(float)

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

scaler = StandardScaler()
X_train_tab = scaler.fit_transform(X_train_tab)
X_val_tab   = scaler.transform(X_val_tab)
X_test_tab  = scaler.transform(X_test_tab)


In [ ]:
# A custom generator that yields Fundus, HVF, OCT, and scaled clinical measurements

def load_image(path):
    img = cv2.imread(path)
    img = cv2.resize(img, IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255.0   # normalise to [0, 1]
    return img

def multimodal_generator(X_paths, X_tabular, y_labels, batch_size, shuffle=True, class_weights=None, use_mixup=False, alpha=0.2):
    n = len(X_paths)
    indices = np.arange(n)
    while True:
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, n, batch_size):
            batch_idx = indices[start : start + batch_size]
            current_batch_size = len(batch_idx)
            
            fundus_batch = []
            hvf_batch    = []
            oct_batch    = []
            tabular_batch = []
            label_batch  = []
            weight_batch = []
            for i in batch_idx:
                fundus_batch.append(load_image(X_paths[i][0]))
                hvf_batch.append(load_image(X_paths[i][1]))
                oct_batch.append(load_image(X_paths[i][2]))
                tabular_batch.append(X_tabular[i])
                lbl = y_labels[i] - 1
                label_batch.append(lbl)
                if class_weights is not None:
                    weight_batch.append(class_weights.get(lbl, 1.0))
            
            f_in = np.array(fundus_batch)
            h_in = np.array(hvf_batch)
            o_in = np.array(oct_batch)
            t_in = np.array(tabular_batch)
            targets = tf.keras.utils.to_categorical(np.array(label_batch), num_classes=3)
            weights = np.array(weight_batch) if class_weights is not None else np.ones(current_batch_size)
            
            if use_mixup and current_batch_size > 1:
                # Sample lambda from Beta distribution
                lam = np.random.beta(alpha, alpha, current_batch_size)
                lam = np.maximum(lam, 1 - lam) # Keep majority label
                
                # Random permutation for mixing
                index = np.random.permutation(current_batch_size)
                
                f_in = lam[:, np.newaxis, np.newaxis, np.newaxis] * f_in + (1 - lam[:, np.newaxis, np.newaxis, np.newaxis]) * f_in[index]
                h_in = lam[:, np.newaxis, np.newaxis, np.newaxis] * h_in + (1 - lam[:, np.newaxis, np.newaxis, np.newaxis]) * h_in[index]
                o_in = lam[:, np.newaxis, np.newaxis, np.newaxis] * o_in + (1 - lam[:, np.newaxis, np.newaxis, np.newaxis]) * o_in[index]
                t_in = lam[:, np.newaxis] * t_in + (1 - lam[:, np.newaxis]) * t_in[index]
                
                targets = lam[:, np.newaxis] * targets + (1 - lam[:, np.newaxis]) * targets[index]
                weights = lam * weights + (1 - lam) * weights[index]
            
            inputs = {
                "fundus_input": f_in,
                "hvf_input":    h_in,
                "oct_input":    o_in,
                "tabular_input": t_in,
            }
            
            if class_weights is not None or use_mixup:
                yield inputs, targets, weights
            else:
                yield inputs, targets

train_gen = multimodal_generator(X_train_img, X_train_tab, y_train, BATCH_SIZE, shuffle=True, class_weights={0: 1.0, 1: 3.0, 2: 1.0}, use_mixup=False)
val_gen   = multimodal_generator(X_val_img,   X_val_tab,   y_val,   BATCH_SIZE, shuffle=False)
test_gen  = multimodal_generator(X_test_img,  X_test_tab,  y_test,  BATCH_SIZE, shuffle=False)

train_steps = int(np.ceil(len(X_train_img) / BATCH_SIZE))
val_steps   = int(np.ceil(len(X_val_img)   / BATCH_SIZE))
test_steps  = int(np.ceil(len(X_test_img)  / BATCH_SIZE))

print(f"Steps per epoch  - train: {train_steps}, val: {val_steps}, test: {test_steps}")


In [ ]:
# ===========================================================================
# 4-MODALITY GLAUCOMA SEVERITY CLASSIFIER
# Architecture overview:
#   Branch 1 (Fundus): EfficientNetV2S encoder + partial U-Net decoder with Attention Gates
#   Branch 2 (HVF):    EfficientNetB0 encoder (separate, different weights)
#   Branch 3 (OCT):    EfficientNetV2S encoder (separate, different weights)
#   Branch 4 (Tabular): MLP branch (Dense -> Batch Normalization -> Dropout -> Dense)
#   Fusion:            GAP outputs from image branches + MLP outputs are concatenated
#   Head:              Dense layers -> softmax over 3 classes
# EfficientNetV2S is pre-trained on ImageNet-21k (21,841 classes vs 1k),
# giving significantly richer features for optic disc/cup and RNFL structures.
# ===========================================================================

def attention_gate(x, g, inter_channels):
    # Soft attention gate (Oktay et al. 2018, Attention U-Net).
    theta_x = Conv2D(inter_channels, (1, 1), padding="same")(x)
    phi_g   = Conv2D(inter_channels, (1, 1), padding="same")(g)
    phi_g   = UpSampling2D(
                  size=(theta_x.shape[1] // phi_g.shape[1],
                        theta_x.shape[2] // phi_g.shape[2]))(phi_g)
    add_xg  = Activation("relu")(Add()([theta_x, phi_g]))
    psi     = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(add_xg)
    return Multiply()([x, psi])

input_shape = (224, 224, 3)

# ── Branch 1: Fundus with EfficientNetB0 + Attention U-Net ──────────────────
fundus_input = Input(shape=input_shape, name="fundus_input")

fundus_weights = "imagenet"
if os.path.exists("fundus_backbone_only.keras"):
    print("Found pre-trained Fundus weights! Using custom medical weights instead of ImageNet.")
    fundus_weights = None
elif os.path.exists("/kaggle/input/fundus-pretrained-weights/fundus_backbone_only.keras"):
    print("Found pre-trained Fundus weights in Kaggle Input! Using custom medical weights.")
    fundus_weights = None

effnet_fundus = EfficientNetV2S(weights=fundus_weights, include_top=False, input_shape=input_shape)

if fundus_weights is None:
    weight_path = "fundus_backbone_only.keras" if os.path.exists("fundus_backbone_only.keras") else "/kaggle/input/fundus-pretrained-weights/fundus_backbone_only.keras"
    effnet_fundus.load_weights(weight_path)

effnet_fundus.name = "effnet_fundus"
effnet_fundus.trainable = True
for layer in effnet_fundus.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
for layer in effnet_fundus.layers[:-30]:  # V2S has more layers than B0, unfreeze last 30
    layer.trainable = False

fundus_features = effnet_fundus(fundus_input)

# U-Net encoder path on fundus input
x = Conv2D(32,  (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(fundus_input)
x = BatchNormalization()(x)
x = Conv2D(32,  (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)
skip1 = x
x = MaxPooling2D((2,2))(x)

x = Conv2D(64,  (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)
x = Conv2D(64,  (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)
skip2 = x
x = MaxPooling2D((2,2))(x)

x = Conv2D(128, (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)
x = Conv2D(128, (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)
skip3 = x
x = MaxPooling2D((2,2))(x)

# Bridge
x = Conv2D(256, (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)
x = Conv2D(256, (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)
bridge = x

# Decoder with Attention Gates
x = UpSampling2D((2,2))(bridge)
att3 = attention_gate(x=skip3, g=bridge, inter_channels=64)
x = Concatenate()([x, att3])
x = Conv2D(128, (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)
x = Conv2D(128, (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)

x = UpSampling2D((2,2))(x)
att2 = attention_gate(x=skip2, g=x, inter_channels=32)
x = Concatenate()([x, att2])
x = Conv2D(64,  (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)
x = Conv2D(64,  (3,3), activation="relu", padding="same", kernel_regularizer=l2(0.0001))(x)
x = BatchNormalization()(x)

unet_features = Conv2D(32, (3,3), activation="relu", padding="same",
                        kernel_regularizer=l2(0.0001))(x)

# GAP from both EfficientNetB0 and U-Net decoder paths
gap_eff_fundus = GlobalAveragePooling2D(name="gap_eff_fundus")(fundus_features)
gap_unet       = GlobalAveragePooling2D(name="gap_unet")(unet_features)

# ── Branch 2: HVF with a separate EfficientNetB0 ────────────────────────────
hvf_input = Input(shape=input_shape, name="hvf_input")

effnet_hvf = EfficientNetB0(weights="imagenet", include_top=False,
                              input_shape=input_shape)
effnet_hvf.name = "effnet_hvf"
effnet_hvf.trainable = True
for layer in effnet_hvf.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
for layer in effnet_hvf.layers[:-20]:
    layer.trainable = False

hvf_features   = effnet_hvf(hvf_input)
gap_eff_hvf    = GlobalAveragePooling2D(name="gap_eff_hvf")(hvf_features)

# ── Branch 3: OCT with a separate EfficientNetB0 ────────────────────────────
oct_input = Input(shape=input_shape, name="oct_input")

oct_weights = "imagenet"
if os.path.exists("oct_backbone_only.keras"):
    print("Found pre-trained OCT weights! Using custom medical weights instead of ImageNet.")
    oct_weights = None
elif os.path.exists("/kaggle/input/oct-pretrained-weights/oct_backbone_only.keras"):
    print("Found pre-trained OCT weights in Kaggle Input! Using custom medical weights.")
    oct_weights = None

effnet_oct = EfficientNetV2S(weights=oct_weights, include_top=False, input_shape=input_shape)

if oct_weights is None:
    weight_path = "oct_backbone_only.keras" if os.path.exists("oct_backbone_only.keras") else "/kaggle/input/oct-pretrained-weights/oct_backbone_only.keras"
    effnet_oct.load_weights(weight_path)
effnet_oct.name = "effnet_oct"
effnet_oct.trainable = True
for layer in effnet_oct.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
for layer in effnet_oct.layers[:-30]:  # V2S has more layers than B0, unfreeze last 30
    layer.trainable = False

oct_features   = effnet_oct(oct_input)
gap_eff_oct    = GlobalAveragePooling2D(name="gap_eff_oct")(oct_features)

# ── Branch 4: Tabular RNFL/GCC Input ─────────────────────────────────────────
tabular_input = Input(shape=(12,), name="tabular_input")
y_tab = Dense(64, activation="relu", kernel_regularizer=l2(0.0001))(tabular_input)
y_tab = BatchNormalization()(y_tab)
y_tab = Dropout(0.3)(y_tab)
y_tab = Dense(32, activation="relu", kernel_regularizer=l2(0.0001))(y_tab)
y_tab = BatchNormalization()(y_tab)

# ── Fusion ───────────────────────────────────────────────────────────────────
# Concatenate all features
merged = Concatenate(name="merged_features")([gap_eff_fundus, gap_unet, gap_eff_hvf, gap_eff_oct, y_tab])

print("Fundus EfficientNetV2S feature vector:", gap_eff_fundus.shape)
print("U-Net decoder feature vector:         ", gap_unet.shape)
print("HVF EfficientNetB0  feature vector:   ", gap_eff_hvf.shape)
print("OCT EfficientNetV2S feature vector:   ", gap_eff_oct.shape)
print("Tabular MLP feature vector:           ", y_tab.shape)
print("Merged feature vector:                ", merged.shape)

# ── Classification head ───────────────────────
x = Dense(256, activation="relu", kernel_regularizer=l2(0.001))(merged)
x = BatchNormalization()(x)
x = Dropout(0.6)(x)
x = Dense(128, activation="relu", kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(64,  activation="relu", kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

# 3 output classes
outputs = Dense(3, activation="softmax")(x)

model = Model(inputs=[fundus_input, hvf_input, oct_input, tabular_input], outputs=outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    # label_smoothing=0.1: prevents overconfident wrong predictions.
    # Critical for Moderate class (low sample count, easily dominated by Mild/Severe).
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

model.summary()


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import math

# ── Why val_loss and not val_accuracy? ──────────────────────────────────────
# The validation set has only ~13 patients (all original, not augmented).
# One misclassified sample = 7.7% swing in val_accuracy, making it extremely noisy.
# val_loss is a continuous, smooth signal — far more stable for EarlyStopping.
# The saved checkpoint will still have the best generalisation.

# ── Cosine Annealing learning rate ──────────────────────────────────────────
# EfficientNetV2S fine-tuning benefits from a smooth decay rather than step-wise ReduceLR.
# We use CosineDecay with a 5-epoch warmup and T_max=100 (full run).
steps_per_epoch_val = train_steps   # re-used for scheduler reference

class WarmupCosineDecay(tf.keras.callbacks.Callback):
    def __init__(self, warmup_epochs, total_epochs, base_lr, min_lr=1e-6):
        super().__init__()
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.base_lr = base_lr
        self.min_lr  = min_lr
        
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.base_lr * (epoch + 1) / self.warmup_epochs
        else:
            import math
            progress = (epoch - self.warmup_epochs) / max(1, self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
            
        # FIXED FOR KERAS 3: Simply assign it directly
        self.model.optimizer.learning_rate = float(lr)

callbacks = [
    EarlyStopping(
        monitor="val_loss",      # val_loss: smooth signal even with 13 val samples
        patience=20,             # extra patience because val_loss improves slowly
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        "Multimodal_Glaucoma_model.keras",
        monitor="val_loss",
        save_best_only=True,
        mode="min",              # lower val_loss = better
        verbose=1
    ),
    WarmupCosineDecay(warmup_epochs=5, total_epochs=EPOCHS, base_lr=1e-4, min_lr=1e-7),
]


In [ ]:
print("Phase 1: Fine-tuning last 30 layers of EfficientNetV2S backbones")
print(f"Training samples: {len(y_train)}  Val samples: {len(y_val)}")
print(f"Class distribution in train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Class distribution in val:   {dict(zip(*np.unique(y_val, return_counts=True)))}")
print(f"Class distribution in test:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

history = model.fit(
    train_gen,
    steps_per_epoch=train_steps,
    epochs=EPOCHS,
    validation_data=val_gen,
    validation_steps=val_steps,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"],     label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.title("Accuracy over epochs")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"],     label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Loss over epochs")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("Multimodal_Glaucoma_model.keras")

val_loss, val_accuracy = model.evaluate(val_gen, steps=val_steps, verbose=0)
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

y_val_pred = []
y_val_true = []

for step in range(val_steps):
    batch_inputs, batch_labels = next(val_gen)
    preds = model.predict(batch_inputs, verbose=0)
    y_val_pred.extend(np.argmax(preds, axis=1))
    y_val_true.extend(np.argmax(batch_labels, axis=1))

y_val_pred = np.array(y_val_pred[:len(X_val_img)])
y_val_true = np.array(y_val_true[:len(X_val_img)])


In [ ]:
import seaborn as sns

class_names = ["Mild", "Moderate", "Severe"]

cm = confusion_matrix(y_val_true, y_val_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - Validation Set")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()

print("Validation Classification Report:")
print(classification_report(y_val_true, y_val_pred, target_names=class_names))


In [ ]:
test_loss, test_accuracy = model.evaluate(test_gen, steps=test_steps, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

y_test_pred = []
y_test_true = []

for step in range(test_steps):
    batch_inputs, batch_labels = next(test_gen)
    preds = model.predict(batch_inputs, verbose=0)
    y_test_pred.extend(np.argmax(preds, axis=1))
    y_test_true.extend(np.argmax(batch_labels, axis=1))

y_test_pred = np.array(y_test_pred[:len(X_test_img)])
y_test_true = np.array(y_test_true[:len(X_test_img)])

cm_test = confusion_matrix(y_test_true, y_test_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_test, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - Test Set")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()

print("Test Classification Report:")
print(classification_report(y_test_true, y_test_pred, target_names=class_names))


In [ ]:
print("==================================================")
print("Phase 2: Hybrid Ensembling (Random Forest on Fine-Tuned Deep Features)")
print("==================================================")

from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import Model

# Extract the fine-tuned feature extractor (before the Dense layers)
feature_extractor = Model(inputs=model.inputs, outputs=model.get_layer("merged_features").output)

def extract_features_and_labels(gen, steps, total_samples):
    features = []
    labels = []
    for step in range(steps):
        out = next(gen)
        batch_inputs = out[0]
        batch_targets = out[1]
        
        # Extract features for the batch
        batch_features = feature_extractor.predict(batch_inputs, verbose=0)
        features.extend(batch_features)
        labels.extend(np.argmax(batch_targets, axis=1))
    return np.array(features[:total_samples]), np.array(labels[:total_samples])

print("Extracting training features...")
# We must use a non-shuffled generator for extraction to align with labels correctly
train_gen_ext = multimodal_generator(X_train_img, X_train_tab, y_train, BATCH_SIZE, shuffle=False)
X_train_features, y_train_ext = extract_features_and_labels(train_gen_ext, train_steps, len(X_train_img))

print("Extracting testing features...")
test_gen_ext = multimodal_generator(X_test_img, X_test_tab, y_test, BATCH_SIZE, shuffle=False)
X_test_features, y_test_ext = extract_features_and_labels(test_gen_ext, test_steps, len(X_test_img))

print("Training Random Forest ensemble on deep features...")
rf = RandomForestClassifier(n_estimators=500, class_weight={0: 1.0, 1: 5.0, 2: 1.0}, random_state=42, n_jobs=-1)
rf.fit(X_train_features, y_train_ext)
rf_preds = rf.predict(X_test_features)
print("\nHybrid Random Forest Classification Report:")
print(classification_report(y_test_ext, rf_preds, target_names=class_names))

from xgboost import XGBClassifier
print("\nTraining Fast XGBoost ensemble on deep features...")
# Heavily weight the Moderate class (label 1) during training
sample_weights = np.array([5.0 if y == 1 else 1.0 for y in y_train_ext])
# max_depth=3 prevents overfitting, n_jobs=-1 makes it run instantly
xgb = XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42, n_jobs=-1)
xgb.fit(X_train_features, y_train_ext, sample_weight=sample_weights)

xgb_preds = xgb.predict(X_test_features)
print("\nHybrid XGBoost Classification Report:")
print(classification_report(y_test_ext, xgb_preds, target_names=class_names))
